# Practical 3: Integrate-and-fire vs. Hodgkin–Huxley (`iaf_psc_alpha` vs `hh_psc_alpha`)

In the previous notebook we characterised the LIF neuron. Here we apply the **same protocols** to the Hodgkin–Huxley model and compare side by side. Each simulation creates both models in the same NEST kernel, so they receive exactly the same stimulus.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import nest

nest.set_verbosity("M_WARNING")
plt.rcParams["figure.dpi"] = 110
print("NEST version:", nest.__version__)

models = ["iaf_psc_alpha", "hh_psc_alpha"]
colors = {"iaf_psc_alpha": "tab:blue", "hh_psc_alpha": "tab:red"}

## Parameters and resting state

The HH model has no threshold parameter and no reset: spikes emerge from the voltage-gated Na⁺ and K⁺ conductances. Note also the different membrane capacitance and the **much faster default synaptic time constant** of `hh_psc_alpha` (0.2 ms vs 2 ms) — we will equalise it in Part 2.

In [ ]:
nest.ResetKernel()
iaf_def = nest.GetDefaults("iaf_psc_alpha")
hh_def = nest.GetDefaults("hh_psc_alpha")

print(f"{'parameter':12s} {'iaf_psc_alpha':>14s} {'hh_psc_alpha':>14s}")
for key in ["C_m", "tau_m", "g_L", "E_L", "V_th", "V_reset", "t_ref", "g_Na", "g_K", "E_Na", "E_K",
            "tau_syn_ex", "tau_syn_in"]:
    v_iaf = iaf_def.get(key, "—")
    v_hh = hh_def.get(key, "—")
    print(f"{key:12s} {str(v_iaf):>14s} {str(v_hh):>14s}")
print("\nHH recordables:", hh_def["recordables"])

Let's simulate both neurons without input to find their resting potentials. The HH resting potential is not $E_L$: it results from the balance of leak, Na⁺ and K⁺ currents.

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

iaf = nest.Create("iaf_psc_alpha")
hh = nest.Create("hh_psc_alpha")
voltmeter = nest.Create("voltmeter", params={"interval": 0.1})
nest.Connect(voltmeter, iaf + hh)
nest.Simulate(200.0)

vm_ev = voltmeter.get("events")
plt.figure(figsize=(7, 3))
for nrn, name in [(iaf, "iaf_psc_alpha"), (hh, "hh_psc_alpha")]:
    mask = vm_ev["senders"] == nrn.global_id
    plt.plot(vm_ev["times"][mask], vm_ev["V_m"][mask], c=colors[name], label=name)
    print(f"{name:14s} resting V_m = {vm_ev['V_m'][mask][-1]:.2f} mV")
plt.xlabel("time (ms)"); plt.ylabel("V_m (mV)"); plt.legend()
plt.show()

The HH neuron needs a few ms to settle from its initial value. In all protocols below we therefore start stimuli at $t = 100$ ms.

## The action potential and the gating variables

We inject the same suprathreshold DC step into both models and record the HH gating variables $m$ (`Act_m`), $h$ (`Inact_h`) and $n$ (`Act_n`).

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

iaf = nest.Create("iaf_psc_alpha")
hh = nest.Create("hh_psc_alpha")
dc = nest.Create("dc_generator", params={"amplitude": 800.0, "start": 100.0, "stop": 300.0})
nest.Connect(dc, iaf + hh)

vm_iaf = nest.Create("voltmeter", params={"interval": 0.1})
vm_hh = nest.Create("voltmeter", params={"interval": 0.1, "record_from": ["V_m", "Act_m", "Inact_h", "Act_n"]})
spike_rec = nest.Create("spike_recorder")
nest.Connect(vm_iaf, iaf)
nest.Connect(vm_hh, hh)
nest.Connect(iaf + hh, spike_rec)
nest.Simulate(300.0)

ev_iaf = vm_iaf.get("events")
ev_hh = vm_hh.get("events")
sp_ev = spike_rec.get("events")
t_first_hh = sp_ev["times"][sp_ev["senders"] == hh.global_id][0]
t_first_iaf = sp_ev["times"][sp_ev["senders"] == iaf.global_id][0]
print(f"first spike: iaf at {t_first_iaf:.1f} ms, HH at {t_first_hh:.1f} ms")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

axes[0].plot(ev_iaf["times"], ev_iaf["V_m"], c=colors["iaf_psc_alpha"], label="iaf")
axes[0].plot(ev_hh["times"], ev_hh["V_m"], c=colors["hh_psc_alpha"], label="HH")
axes[0].set_xlim(80, 300); axes[0].set_xlabel("time (ms)"); axes[0].set_ylabel("V_m (mV)")
axes[0].set_title("DC 800 pA"); axes[0].legend()

win = (ev_hh["times"] > t_first_hh - 3) & (ev_hh["times"] < t_first_hh + 12)
axes[1].plot(ev_hh["times"][win] - t_first_hh, ev_hh["V_m"][win], c=colors["hh_psc_alpha"])
axes[1].set_xlabel("time from spike (ms)"); axes[1].set_ylabel("V_m (mV)"); axes[1].set_title("HH action potential")

axes[2].plot(ev_hh["times"][win] - t_first_hh, ev_hh["Act_m"][win], label="m (Na act.)")
axes[2].plot(ev_hh["times"][win] - t_first_hh, ev_hh["Inact_h"][win], label="h (Na inact.)")
axes[2].plot(ev_hh["times"][win] - t_first_hh, ev_hh["Act_n"][win], label="n (K act.)")
axes[2].set_xlabel("time from spike (ms)"); axes[2].set_ylabel("gating"); axes[2].set_title("HH gating variables")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

v_spike = ev_hh["V_m"][win]
print(f"HH spike peak = {v_spike.max():.1f} mV,  after-hyperpolarisation minimum = {v_spike.min():.1f} mV")

## DC steps: same current, different responses

Both models receive identical current steps (one `dc_generator` per amplitude, connected to one iaf and one HH neuron).

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

dc_amplitudes = [200.0, 400.0, 700.0, 1500.0]   # pA
t_on, t_off, t_sim = 100.0, 500.0, 600.0

iaf_pop = nest.Create("iaf_psc_alpha", len(dc_amplitudes))
hh_pop = nest.Create("hh_psc_alpha", len(dc_amplitudes))
dc_gens = nest.Create("dc_generator", len(dc_amplitudes))
dc_gens.set([{"amplitude": a, "start": t_on, "stop": t_off} for a in dc_amplitudes])
nest.Connect(dc_gens, iaf_pop, "one_to_one")
nest.Connect(dc_gens, hh_pop, "one_to_one")

voltmeter = nest.Create("voltmeter", params={"interval": 0.1})
spike_rec = nest.Create("spike_recorder")
nest.Connect(voltmeter, iaf_pop + hh_pop)
nest.Connect(iaf_pop + hh_pop, spike_rec)
nest.Simulate(t_sim)

vm_ev = voltmeter.get("events")
sp_ev = spike_rec.get("events")

fig, axes = plt.subplots(len(dc_amplitudes), 2, figsize=(12, 9), sharex=True)
for row, amp in enumerate(dc_amplitudes):
    for col, (pop, name) in enumerate([(iaf_pop, "iaf_psc_alpha"), (hh_pop, "hh_psc_alpha")]):
        gid = pop[row].global_id
        mask = vm_ev["senders"] == gid
        spikes = sp_ev["times"][sp_ev["senders"] == gid]
        ax = axes[row, col]
        ax.plot(vm_ev["times"][mask], vm_ev["V_m"][mask], c=colors[name], lw=0.8)
        if name == "iaf_psc_alpha":
            ax.vlines(spikes, -55, -40, color="k", lw=0.8)
        ax.axvspan(t_on, t_off, color="orange", alpha=0.08)
        ax.set_title(f"{name}, I = {amp:.0f} pA: {len(spikes)} spikes", fontsize=9)
        ax.set_ylabel("V_m (mV)")
axes[-1, 0].set_xlabel("time (ms)"); axes[-1, 1].set_xlabel("time (ms)")
plt.tight_layout()
plt.show()

## f–I curves: type I vs. type II excitability

We sweep the bias current `I_e` for both models and count spikes only after 500 ms, i.e. we measure the **sustained** firing rate. We also count the spikes in the first 500 ms, to capture onset responses.

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

fi_amplitudes = np.sort(np.concatenate([np.arange(0.0, 2001.0, 20.0),     # coarse sweep
                                        np.arange(376.0, 400.0, 1.0)]))   # fine points just above the LIF rheobase
t_start_count, t_sim = 500.0, 2500.0
t_count = (t_sim - t_start_count) / 1000.0

iaf_pop = nest.Create("iaf_psc_alpha", len(fi_amplitudes))
hh_pop = nest.Create("hh_psc_alpha", len(fi_amplitudes))
iaf_pop.set(I_e=fi_amplitudes.tolist())
hh_pop.set(I_e=fi_amplitudes.tolist())

sr_sustained = nest.Create("spike_recorder", params={"start": t_start_count})
sr_all = nest.Create("spike_recorder")
nest.Connect(iaf_pop + hh_pop, sr_sustained)
nest.Connect(iaf_pop + hh_pop, sr_all)
nest.Simulate(t_sim)

s_sus = sr_sustained.get("events")["senders"]
s_all = sr_all.get("events")["senders"]

rates = {}
any_spike = {}
for pop, name in [(iaf_pop, "iaf_psc_alpha"), (hh_pop, "hh_psc_alpha")]:
    rates[name] = np.array([np.sum(s_sus == gid) for gid in pop.tolist()]) / t_count
    any_spike[name] = np.array([np.sum(s_all == gid) > 0 for gid in pop.tolist()])

plt.figure(figsize=(6.5, 4.5))
for name in models:
    plt.plot(fi_amplitudes, rates[name], "o-", ms=3, c=colors[name], label=name)
plt.xlabel("I_e (pA)")
plt.ylabel("sustained firing rate (spikes/s)")
plt.legend()
plt.title("f–I curves")
plt.show()

for name in models:
    I_single = fi_amplitudes[np.argmax(any_spike[name])]
    I_rep = fi_amplitudes[np.argmax(rates[name] > 0)]
    f_min = rates[name][rates[name] > 0].min()
    print(f"{name:14s}: first spike at I >= {I_single:.0f} pA, repetitive firing at I >= {I_rep:.0f} pA, "
          f"lowest sustained rate = {f_min:.1f} Hz")

## 1.5 Rebound (anode-break) excitation

We now apply a **hyperpolarising** current for 200 ms and then release it. What would you expect from a neuron that only integrates its input?

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

hyp_amplitudes = [-100.0, -200.0, -400.0, -800.0]   # pA
iaf_pop = nest.Create("iaf_psc_alpha", len(hyp_amplitudes))
hh_pop = nest.Create("hh_psc_alpha", len(hyp_amplitudes))
steps = nest.Create("step_current_generator", len(hyp_amplitudes))
steps.set([{"amplitude_times": [100.0, 300.0], "amplitude_values": [a, 0.0]} for a in hyp_amplitudes])
nest.Connect(steps, iaf_pop, "one_to_one")
nest.Connect(steps, hh_pop, "one_to_one")

vm_hh = nest.Create("voltmeter", params={"interval": 0.1, "record_from": ["V_m", "Inact_h", "Act_n"]})
vm_iaf = nest.Create("voltmeter", params={"interval": 0.1})
spike_rec = nest.Create("spike_recorder")
nest.Connect(vm_hh, hh_pop)
nest.Connect(vm_iaf, iaf_pop)
nest.Connect(iaf_pop + hh_pop, spike_rec)
nest.Simulate(450.0)

ev_hh = vm_hh.get("events")
ev_iaf = vm_iaf.get("events")
sp_ev = spike_rec.get("events")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, amp in enumerate(hyp_amplitudes):
    m_iaf = ev_iaf["senders"] == iaf_pop[i].global_id
    m_hh = ev_hh["senders"] == hh_pop[i].global_id
    n_hh = np.sum(sp_ev["senders"] == hh_pop[i].global_id)
    n_iaf = np.sum(sp_ev["senders"] == iaf_pop[i].global_id)
    axes[0].plot(ev_iaf["times"][m_iaf], ev_iaf["V_m"][m_iaf], lw=0.8, label=f"{amp:.0f} pA ({n_iaf} spk)")
    axes[1].plot(ev_hh["times"][m_hh], ev_hh["V_m"][m_hh], lw=0.8, label=f"{amp:.0f} pA ({n_hh} spk)")
    axes[2].plot(ev_hh["times"][m_hh], ev_hh["Inact_h"][m_hh], lw=0.8, label=f"h, {amp:.0f} pA")

axes[0].set_title("iaf_psc_alpha"); axes[1].set_title("hh_psc_alpha"); axes[2].set_title("HH Na⁺ inactivation h")
for ax in axes:
    ax.set_xlabel("time (ms)"); ax.axvspan(100, 300, color="tab:purple", alpha=0.06); ax.legend(fontsize=7)
axes[0].set_ylabel("V_m (mV)"); axes[1].set_ylabel("V_m (mV)")
plt.tight_layout()
plt.show()

# Computational cost

The HH model integrates four coupled non-linear ODEs with an adaptive solver; the LIF model is solved exactly with a linear propagator.
We simulate 1000 neurons of each type for 1 s of biological time and measure the wall-clock time.

In [ ]:
n_neurons = 1000
t_sim = 1000.0
wall_times = {}
n_spikes = {}

for name in models:
    nest.ResetKernel()
    nest.resolution = 0.1
    pop = nest.Create(name, n_neurons)
    pop.set(I_e=800.0)
    spike_rec = nest.Create("spike_recorder")
    nest.Connect(pop, spike_rec)

    t0 = time.perf_counter()
    nest.Simulate(t_sim)
    wall_times[name] = time.perf_counter() - t0
    n_spikes[name] = spike_rec.get("n_events")

for name in models:
    print(f"{name:14s}: {wall_times[name]:.3f} s wall-clock, {n_spikes[name]} spikes")
print(f"HH / LIF cost ratio: {wall_times['hh_psc_alpha'] / wall_times['iaf_psc_alpha']:.1f}x")